# Translate FR Queries to EN - Kaggle

Creates `fr_train.json`, `fr_dev.json`, `fr_test.json` in `/kaggle/working/.cache/translations2/` using the same NLLB/Marian approach as the existing translation pipeline.


In [ ]:
# -- 0. GPU check -----------------------------------------------------------
import subprocess
try:
    r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(r.stdout if r.returncode == 0 else 'No GPU found')
except FileNotFoundError:
    print('GPU is not enabled. Turn on Settings -> Accelerator -> GPU.')


In [ ]:
# -- 1. Install + restart kernel -------------------------------------------
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'datasets', 'transformers', 'sentencepiece'])
print('Restarting kernel...')
import IPython
IPython.Application.instance().kernel.do_shutdown(True)


In [ ]:
# -- 2. Config --------------------------------------------------------------
import torch
from pathlib import Path

DATASET_NAME = 'sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims'
LANG = 'fr'
SPLITS = ['train', 'dev', 'test']
BACKEND = 'nllb'   # 'nllb' or 'marian'
MODEL_NAME = None  # None -> default backend model
BATCH_SIZE = 32
NUM_BEAMS = 4
MAX_INPUT_LENGTH = 256
MAX_NEW_TOKENS = 256
SOURCE_PRECLEAN = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR = Path('/kaggle/working/.cache/translations2')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('device:', DEVICE)
print('output:', OUTPUT_DIR)


In [ ]:
# -- 3. Translator ----------------------------------------------------------
import json
import re
from dataclasses import dataclass

import torch
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, MarianMTModel, MarianTokenizer

_UNICODE_TABLE = str.maketrans({
    '\u2018': "'", '\u2019': "'", '\u201c': '"', '\u201d': '"', '\u2026': '...',
    '\u2011': '-', '\u2013': '-', '\u2014': ' ',
    '\u2066': '', '\u2067': '', '\u2068': '', '\u2069': '',
    '\u200b': '', '\u200c': '', '\u200d': '', '\ufeff': '',
})
_URL_RE = re.compile(r'https?://\S+|www\.\S+', re.IGNORECASE)
_LEADING_MENTION_CHAIN_RE = re.compile(r'^\s*(?:@\w+[\s,;:!.\-]*){2,}')
_HASHTAG_RE = re.compile(r'#(\w+)')
_TRAILING_SOURCE_RE = re.compile(r'\s+(?:quelle|source|via|mehr dazu|en savoir plus|lire ici)\s*[:\-].*$', re.IGNORECASE)
_CAMEL_SPLIT_1_RE = re.compile(r'([a-z])([A-Z])')
_CAMEL_SPLIT_2_RE = re.compile(r'([A-Z]+)([A-Z][a-z])')


def _split_camel_case(token: str) -> str:
    token = _CAMEL_SPLIT_2_RE.sub(r'\1 \2', token)
    token = _CAMEL_SPLIT_1_RE.sub(r'\1 \2', token)
    return token


def _expand_hashtag_token(match):
    token = match.group(1).replace('_', ' ')
    return _split_camel_case(token)


def preprocess_translation_input(text: str, lang: str) -> str:
    text = text.translate(_UNICODE_TABLE)
    text = _URL_RE.sub(' ', text)
    text = _LEADING_MENTION_CHAIN_RE.sub(' ', text)
    text = _HASHTAG_RE.sub(_expand_hashtag_token, text)
    if lang == 'fr':
        text = _TRAILING_SOURCE_RE.sub('', text)
    return re.sub(r'\s+', ' ', text).strip()


MARIAN_MODELS = {
    'fr': 'Helsinki-NLP/opus-mt-fr-en',
}
NLLB_MODEL = 'facebook/nllb-200-distilled-600M'
NLLB_SOURCE_LANG = {'fr': 'fra_Latn'}
NLLB_TARGET_LANG = 'eng_Latn'


@dataclass
class TranslationConfig:
    backend: str
    batch_size: int
    device: str
    num_beams: int
    max_input_length: int
    max_new_tokens: int
    source_preclean: bool
    model_name: str | None


class Translator:
    def __init__(self, lang: str, cfg: TranslationConfig):
        self.lang = lang
        self.cfg = cfg
        self.device = cfg.device
        self.backend = cfg.backend

        if self.backend == 'marian':
            model_name = cfg.model_name or MARIAN_MODELS[lang]
            print(f'Loading Marian model: {model_name}')
            self.tokenizer = MarianTokenizer.from_pretrained(model_name)
            self.model = MarianMTModel.from_pretrained(model_name).to(self.device)
            self._forced_bos_token_id = None
        elif self.backend == 'nllb':
            model_name = cfg.model_name or NLLB_MODEL
            print(f'Loading NLLB model: {model_name}')
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(self.device)
            self.tokenizer.src_lang = NLLB_SOURCE_LANG[lang]
            self._forced_bos_token_id = self.tokenizer.convert_tokens_to_ids(NLLB_TARGET_LANG)
        else:
            raise ValueError(f'Unsupported backend: {self.backend}')

        self.model.eval()

    def preprocess(self, text: str) -> str:
        if not self.cfg.source_preclean:
            return text
        return preprocess_translation_input(text, self.lang)

    def translate_batch(self, texts: list[str]) -> list[str]:
        inputs = self.tokenizer(
            texts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=self.cfg.max_input_length,
        ).to(self.device)
        gen_kwargs = {
            'num_beams': self.cfg.num_beams,
            'max_new_tokens': self.cfg.max_new_tokens,
            'early_stopping': True,
        }
        if self._forced_bos_token_id is not None:
            gen_kwargs['forced_bos_token_id'] = self._forced_bos_token_id
        with torch.no_grad():
            outputs = self.model.generate(**inputs, **gen_kwargs)
        return self.tokenizer.batch_decode(outputs, skip_special_tokens=True)

    def translate_all(self, texts: list[str]) -> tuple[list[str], list[str]]:
        cleaned = [self.preprocess(text) for text in texts]
        translated = []
        for i in tqdm(range(0, len(cleaned), self.cfg.batch_size), desc=f'Translating {self.lang}->en ({self.backend})'):
            batch = cleaned[i:i + self.cfg.batch_size]
            translated.extend(self.translate_batch(batch))
        return cleaned, translated


def load_split_data(lang: str, split: str):
    return load_dataset(DATASET_NAME, lang)[split]


In [ ]:
# -- 4. Translate + save ----------------------------------------------------
cfg = TranslationConfig(
    backend=BACKEND,
    batch_size=BATCH_SIZE,
    device=DEVICE,
    num_beams=NUM_BEAMS,
    max_input_length=MAX_INPUT_LENGTH,
    max_new_tokens=MAX_NEW_TOKENS,
    source_preclean=SOURCE_PRECLEAN,
    model_name=MODEL_NAME,
)

translator = Translator(LANG, cfg)

for split in SPLITS:
    print(f'\n=== {LANG.upper()} / {split} ===')
    data = load_split_data(LANG, split)
    texts = list(data['text'])
    indices = list(data['index'])

    cleaned, translated = translator.translate_all(texts)
    output = {str(idx): tr for idx, tr in zip(indices, translated)}
    out_path = OUTPUT_DIR / f'{LANG}_{split}.json'
    out_path.write_text(json.dumps(output, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'Saved {len(output)} translations -> {out_path}')

    empties = sum(1 for v in output.values() if not str(v).strip())
    print(f'Empty translations: {empties}')
    for i in range(min(3, len(texts))):
        print(f'[{indices[i]}] raw:     {texts[i][:120]}')
        print(f'     cleaned: {cleaned[i][:120]}')
        print(f'     english: {translated[i][:120]}')


In [ ]:
# -- 5. Package output ------------------------------------------------------
import zipfile

zip_path = Path('/kaggle/working/translations2_fr.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in OUTPUT_DIR.glob('fr_*.json'):
        zf.write(path, arcname=path.relative_to(OUTPUT_DIR.parent))

print('Created:', zip_path)
for path in OUTPUT_DIR.glob('fr_*.json'):
    print(path, f'{path.stat().st_size / 1e6:.2f} MB')
